In [17]:
import pandas as pd
from pathlib import Path
import json

In [3]:
hgsoc_tcga_gdc_clinical_cleaned = pd.read_csv("../data/processed/hgsoc_tcga_gdc_clinical_cleaned.csv")
tcga_ov_imaging_cleaned = pd.read_csv("../data/processed/tcga_ov_imaging_cleaned.csv")

In [10]:
hgsoc_tcga_gdc_clinical_cleaned['patient_id_canonical'] = hgsoc_tcga_gdc_clinical_cleaned['patient_id'].str.upper().str.strip()
tcga_ov_imaging_cleaned['patient_id_canonical'] = tcga_ov_imaging_cleaned['patient_id'].str.upper().str.strip()

In [11]:
universal_df = pd.merge(
    hgsoc_tcga_gdc_clinical_cleaned,
    tcga_ov_imaging_cleaned,
    on='patient_id_canonical',
    how='inner',  
    suffixes=('_clinical', '_imaging')
)

In [12]:
num_clinical = hgsoc_tcga_gdc_clinical_cleaned['patient_id_canonical'].nunique()
num_imaging = tcga_ov_imaging_cleaned['patient_id_canonical'].nunique()
num_matched = universal_df['patient_id_canonical'].nunique()

match_rate = num_matched / min(num_clinical, num_imaging) * 100

print(f"Clinical patients: {num_clinical}")
print(f"Imaging patients: {num_imaging}")
print(f"Matched patients: {num_matched}")
print(f"Match rate: {match_rate:.2f}%")

Clinical patients: 588
Imaging patients: 143
Matched patients: 138
Match rate: 96.50%


In [15]:
processed_path = Path("../data/processed")

In [16]:
universal_df.to_csv(processed_path / "ovarian_universal.csv", index=False)

In [28]:
schema = []

for col in universal_df.columns:
    col_info = {
        "column_name": col,
        "dtype": str(universal_df[col].dtype),
        "description": "",  
        "allowed_values": None,
        "provenance": (
            "clinical" if "_clinical" in col else
            "imaging" if "_imaging" in col else
            "universal"
        )
    }
    
    if universal_df[col].dtype == "object" or universal_df[col].dtype.name == "category":
        unique_vals = universal_df[col].dropna().unique()
        if len(unique_vals) <= 20:
            col_info["allowed_values"] = unique_vals.tolist()
    
    schema.append(col_info)

with open("../schema.json", "w") as f:
    json.dump(schema, f, indent=4)

print(f"Schema generated for {len(schema)} columns!")

Schema generated for 77 columns!


In [29]:
from datetime import datetime

metadata = {
    "dataset_name": "Ovarian Universal Dataset",
    "description": "Integrated patient-level dataset combining HGSOC TCGA clinical and TCGA-OV imaging metadata.",
    "sources": [
        {
            "name": "HGSOC TCGA GDC Clinical",
            "url": "https://www.cbioportal.org/study/clinicalData?id=hgsoc_tcga_gdc",
            "license": "Open access",
            "citation": "TCGA Research Network: https://www.cancer.gov/tcga"
        },
        {
            "name": "TCGA-OV Imaging",
            "url": "https://www.cancerimagingarchive.net/collection/tcga-ov/",
            "license": "CC BY 3.0",
            "citation": "Clark K et al. The Cancer Imaging Archive (TCIA). Nat Rev Cancer 2013."
        }
    ],
    "processing_steps": [
        "Loaded raw clinical and imaging datasets",
        "Standardized column names to lowercase with underscores",
        "Dropped duplicate rows",
        "Created patient_id_canonical key for linking",
        "Merged clinical and imaging datasets using inner join on patient_id_canonical",
        "Saved final dataset as ovarian_universal.csv"
    ],
    "date_created": datetime.today().strftime("%Y-%m-%d"),
    "version": "v1.0"
}

with open("../metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved to data/processed/metadata.json")

Metadata saved to data/processed/metadata.json
